# 11 SVC With Basic + Connected-Component Features

This notebook extracts connected-component features from bright particle-like regions, combines them with existing basic image features, and tunes SVC on the combined engineered feature set.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

## 2. Imports

In [ ]:
from src.extract_basic_features import BASIC_FEATURE_COLUMNS
from src.extract_component_features import (
    BASIC_FEATURES_CSV,
    COMBINED_FEATURES_CSV,
    COMPONENT_FEATURES_CSV,
    RESIZE_SHAPE,
    combine_basic_and_component_features,
    component_feature_columns,
    load_basic_features,
    load_metadata,
    load_or_build_component_features,
    save_combined_features,
    zero_component_count,
)
from src.train_svc_components import (
    BEST_PARAMS_JSON,
    CONFUSION_MATRIX_FIGURE,
    SCORES_CSV,
    build_comparison_table,
    feature_columns,
    print_previous_svc_comparison,
    train_svc_components,
)

## 3. Paths and configuration

In [ ]:
print(f"Basic features CSV: {BASIC_FEATURES_CSV}")
print(f"Component features CSV: {COMPONENT_FEATURES_CSV}")
print(f"Combined features CSV: {COMBINED_FEATURES_CSV}")
print(f"SVC scores CSV: {SCORES_CSV}")
print(f"Confusion matrix figure: {CONFUSION_MATRIX_FIGURE}")
print(f"Best params JSON: {BEST_PARAMS_JSON}")
print(f"Component resize shape: {RESIZE_SHAPE}")

## 4. Load metadata

In [ ]:
metadata = load_metadata()
metadata.head()

## 5. Sanity checks

In [ ]:
print(f"Metadata rows: {len(metadata)}")
print(metadata["label"].value_counts().sort_index())
assert metadata["area_group"].notna().all()
assert metadata["filename"].is_unique

## 6. Main analysis

Connected-component features approximate particle/blob and aggregate structure. Metadata remains only for tracking, labels, auditing, and grouped splitting.

In [ ]:
component_features = load_or_build_component_features(metadata)
basic_features = load_basic_features()
combined = combine_basic_and_component_features(basic_features, component_features)
combined_path = save_combined_features(combined)

print(f"Component feature table shape: {component_features.shape}")
print(f"Combined feature table shape: {combined.shape}")
print(f"Component columns: {len(component_feature_columns(component_features))}")
print(f"Images with zero detected components: {zero_component_count(component_features)}")
print(f"Saved combined features to {combined_path}")
combined.head()

## 7. Results/figures

Compare basic-only tuned SVC versus basic+components tuned SVC using macro F1, balanced accuracy, and disordered recall.

In [ ]:
scores, report, matrix, figure_path, params_path, y_train, y_test, search = train_svc_components(combined)
scores.to_csv(SCORES_CSV, index=False)

print("Per-class precision/recall/F1:")
print(report)
print("Confusion matrix rows=true, columns=predicted:")
print(matrix)
print("\nBest parameters:")
print(search.best_params_)
print(f"Best CV macro F1: {search.best_score_:.4f}")
print_previous_svc_comparison(scores)
print("\nModel comparison:")
print(build_comparison_table(scores))
print(f"\nSaved scores to {SCORES_CSV}")
print(f"Saved confusion matrix figure to {figure_path}")
print(f"Saved best parameters to {params_path}")
scores

## 8. Notes for report

- Connected-component features approximate particle/blob and aggregate structure.
- These features are physically meaningful for SEM nanoparticle images.
- They may capture differences in particle density, aggregation, and blob-size distribution.
- SVC is used because it performed best on the basic feature set.
- Compare basic SVC versus basic+components SVC using macro F1, balanced accuracy, and disordered recall.
- If connected-component features do not improve performance, state that they did not add useful signal beyond the basic features on this split.